# Module 2 — Emotion Classifier Inference
### RAG-Based Mental Health Support Chatbot

**Goal:** Load the trained emotion classifier models and run inference locally.

---
## 0. Install & Import Dependencies

In [1]:
!pip install torch transformers safetensors --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [2]:
import re
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DistilBertTokenizerFast, DistilBertForSequenceClassification

---
## 1. Configuration

In [3]:
DEVICE = torch.device('cpu')

# Must match training configuration
EMBED_DIM      = 100
RNN_HIDDEN_DIM = 256
RNN_NUM_LAYERS = 2
RNN_DROPOUT    = 0.3
NUM_CLASSES    = 6
BERT_MAX_LEN   = 64
MENTAL_ROBERTA_MAX_LEN = 128
MENTAL_ROBERTA_MODEL_DIR = r'../Mahmoud Saqr/models/emotion_classifier'

label2emotion = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

print(f"Device: {DEVICE}")

Device: cpu


---
## 2. Rebuild Vocabulary

> The vocabulary must be rebuilt exactly as during training to ensure correct token-to-index mapping.

In [7]:
from datasets import load_dataset
from collections import Counter
import re

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r"[^a-z0-9'\s]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load dataset to rebuild vocab from training data
ds_split   = load_dataset('dair-ai/emotion', 'split')
ds_unsplit = load_dataset('dair-ai/emotion', 'unsplit')

from datasets import concatenate_datasets
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

SEED = 42
all_splits = list(ds_split.values()) + list(ds_unsplit.values())
combined   = concatenate_datasets(all_splits)
df         = combined.to_pandas().drop_duplicates(subset='text').reset_index(drop=True)
df['text_clean'] = df['text'].apply(clean_text)

train_df, temp_df = train_test_split(df, test_size=0.1, stratify=df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)

# Build vocab
MIN_FREQ   = 2
PAD_TOKEN  = '<PAD>'
UNK_TOKEN  = '<UNK>'

word_freq = Counter()
for text in train_df['text_clean']:
    word_freq.update(text.split())

vocab    = [PAD_TOKEN, UNK_TOKEN] + [w for w, f in word_freq.items() if f >= MIN_FREQ]
word2idx = {word: idx for idx, word in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
PAD_IDX    = word2idx[PAD_TOKEN]
UNK_IDX    = word2idx[UNK_TOKEN]
MAX_LEN    = int(np.percentile(train_df['text_clean'].str.split().str.len(), 95))

print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"MAX_LEN         : {MAX_LEN}")

Vocabulary size : 36753
MAX_LEN         : 41


---
## 3. Model Definition

In [8]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, rnn_type='RNN',
                 pretrained_embeddings=None, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if pretrained_embeddings is not None:
            self.embedding.weight = nn.Parameter(pretrained_embeddings)
        self.dropout = nn.Dropout(dropout)
        rnn_cls = {'RNN': nn.RNN, 'GRU': nn.GRU, 'LSTM': nn.LSTM}[rnn_type]
        self.rnn = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.rnn_type = rnn_type
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.rnn(embedded) if self.rnn_type != 'LSTM' \
                         else self.rnn(embedded)[:2]
        if self.rnn_type == 'LSTM':
            hidden = hidden[0]
        last_hidden = hidden[-1]
        return self.fc(last_hidden)

print("RNNClassifier defined ✓")

RNNClassifier defined ✓


---
## 4. Load Models

In [9]:
def load_rnn_model(rnn_type: str) -> RNNClassifier:
    model = RNNClassifier(
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        hidden_dim=RNN_HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        num_layers=RNN_NUM_LAYERS,
        dropout=RNN_DROPOUT,
        rnn_type=rnn_type,
        pad_idx=PAD_IDX
    ).to(DEVICE)
    model.load_state_dict(torch.load(f'./artifacts/{rnn_type.lower()}_emotion.pt', map_location=DEVICE))
    model.eval()
    print(f"{rnn_type} loaded ✓")
    return model

rnn_model  = load_rnn_model('RNN')
gru_model  = load_rnn_model('GRU')
lstm_model = load_rnn_model('LSTM')

bert_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert_emotion')
bert_model     = DistilBertForSequenceClassification.from_pretrained('distilbert_emotion').to(DEVICE)
bert_model.eval()

mental_roberta_tokenizer = AutoTokenizer.from_pretrained(MENTAL_ROBERTA_MODEL_DIR)
mental_roberta_model     = AutoModelForSequenceClassification.from_pretrained(MENTAL_ROBERTA_MODEL_DIR).to(DEVICE)
mental_roberta_model.eval()
print("Mental RoBERTa loaded")
print("DistilBERT loaded ✓")

rnn_results = {
    'RNN':  {'model': rnn_model},
    'GRU':  {'model': gru_model},
    'LSTM': {'model': lstm_model},
}

RNN loaded ✓
GRU loaded ✓
LSTM loaded ✓


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Mental RoBERTa loaded
DistilBERT loaded ✓


---
## 5. Inference Utilities

In [10]:
def text_to_indices(text: str, word2idx: dict, max_len: int) -> list:
    tokens  = text.split()[:max_len]
    indices = [word2idx.get(t, UNK_IDX) for t in tokens]
    indices += [PAD_IDX] * (max_len - len(indices))
    return indices


def predict_emotion(text: str, model, is_transformer: bool, tokenizer=None, max_len=BERT_MAX_LEN):
    model.eval()
    with torch.no_grad():
        if is_transformer:
            if tokenizer is None:
                raise ValueError('A tokenizer is required for transformer models.')
            encoding = tokenizer(
                text, max_length=max_len,
                padding='max_length', truncation=True, return_tensors='pt'
            )
            input_ids      = encoding['input_ids'].to(DEVICE)
            attention_mask = encoding['attention_mask'].to(DEVICE)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        else:
            clean   = clean_text(text)
            indices = text_to_indices(clean, word2idx, MAX_LEN)
            tensor  = torch.tensor([indices], dtype=torch.long).to(DEVICE)
            logits  = model(tensor)

        probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
        pred  = probs.argmax()

    return label2emotion[pred], {label2emotion[i]: float(p) for i, p in enumerate(probs)}


print("Inference utilities defined ✓")

Inference utilities defined ✓


---
## 6. Inference Demo

In [12]:
test_sentences = [
    # Sadness
    "I feel so hopeless and empty, nothing brings me joy anymore.",
    "I've been crying all day and I don't know why.",
    "Everything feels heavy, and I can barely get out of bed.",
    "I miss how happy I used to feel before everything changed.",
    "No matter what I do, this sadness keeps following me around.",
    # Joy
    "I'm so excited about my progress today, feeling great!",
    "Today was the best day of my life, I'm on top of the world!",
    "I finally finished something important, and I feel proud of myself.",
    "I woke up smiling and full of energy for the first time in weeks.",
    "Spending time with my friends today made me feel genuinely happy.",
    # Fear
    "I'm terrified of what might happen next, I can't stop shaking.",
    "I'm so scared, I don't feel safe at all.",
    "My chest tightens whenever I think about tomorrow's appointment.",
    "I keep imagining the worst outcome, and it makes me panic.",
    "Every small noise makes me jump because I feel so anxious.",
    # Love
    "I love spending time with my family, it warms my heart.",
    "I feel so grateful for the people around me, they mean everything.",
    "My partner's kindness makes me feel deeply cared for and understood.",
    "I feel connected and safe when my closest friends check on me.",
    "Thinking about my sister fills me with affection and appreciation.",
    # Anger
    "This situation makes me so angry, I can't stand it anymore.",
    "I'm furious, nobody ever listens to me.",
    "I feel irritated because people keep ignoring my boundaries.",
    "It makes my blood boil when I am treated so unfairly.",
    "I'm tired of being blamed for things that were not my fault.",
    # Surprise
    "I didn't expect that at all, completely caught off guard.",
    "I can't believe what just happened, this is shocking.",
    "That message came out of nowhere and left me speechless.",
    "I was stunned when they announced the news so suddenly.",
    "I never saw this coming, and I am still trying to process it.",
]

all_models = {
    'RNN':             (rnn_results['RNN']['model'], False, None, None),
    'GRU':             (rnn_results['GRU']['model'], False, None, None),
    'LSTM':            (rnn_results['LSTM']['model'], False, None, None),
    'DistilBERT':      (bert_model, True, bert_tokenizer, BERT_MAX_LEN),
    'Mental RoBERTa':  (mental_roberta_model, True, mental_roberta_tokenizer, MENTAL_ROBERTA_MAX_LEN),
}

header = f"{'Sentence':<55}" + ''.join(f"{model_name:<16}" for model_name in all_models)
print(header)
print("-" * len(header))

for sent in test_sentences:
    row = f"{sent[:52]:<55}"
    for model_name, (model, is_transformer, tokenizer, max_len) in all_models.items():
        emotion, _ = predict_emotion(sent, model, is_transformer, tokenizer, max_len or BERT_MAX_LEN)
        row += f"{emotion:<16}"
    print(row)

Sentence                                               RNN             GRU             LSTM            DistilBERT      Mental RoBERTa  
---------------------------------------------------------------------------------------------------------------------------------------
I feel so hopeless and empty, nothing brings me joy    sadness         sadness         sadness         sadness         sadness         
I've been crying all day and I don't know why.         anger           fear            anger           sadness         sadness         
Everything feels heavy, and I can barely get out of    anger           anger           joy             sadness         sadness         
I miss how happy I used to feel before everything ch   anger           anger           sadness         sadness         joy             
No matter what I do, this sadness keeps following me   anger           anger           anger           sadness         sadness         
I'm so excited about my progress today, feeling 